In [1]:
import os
import shutil
import re
import requests
import json
import nest_asyncio
from pathlib import Path
from bs4 import BeautifulSoup
from datetime import datetime
from langdetect import detect
from tqdm.notebook import tqdm
from tika import parser
import pandas as pd

/home/saskya/dev/tb/polylex-chatbot/.venv/lib/python3.12/site-packages/tika/__init__.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


In [2]:
from rag.lib.metadata import build_metadata
from rag.lib.config import LEXES_API_URL, LANGUAGES

In [3]:
DATA_DIR = Path("data")
STATS_DIR = Path("stats")

In [4]:
nest_asyncio.apply()

response = requests.get(LEXES_API_URL)
if response.status_code != 200:
    # TODO : a gerer dans les logs
    raise Exception(f"Unexpected status code while fetching : {response.status_code}")

data = build_metadata(response)

No SPARQL results for https://fedlex.data.admin.ch/redirectInfo?url=eli%2Fcc%2F1993%2F210_210_210%2Ffr&original=https:%2F%2Fwww.admin.ch%2Fch%2Ff%2Frs%2Fc414_110.html


In [5]:
# add unique language for each entry

hard_coded_langs = {
    "https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/1.1.7-Ruling-EPFL-Geneve.pdf": "fr",
    "https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/1.1.7-Ruling-EPFL-Vaud-.pdf": "fr",
    "https://fedlex.data.admin.ch/filestore/fedlex.data.admin.ch/eli/cc/2008/857/20250501/fr/pdf-a/fedlex-data-admin-ch-eli-cc-2008-857-20250501-fr-pdf-a-1.pdf": "fr",
    "https://ethrat.stage.mxm.agency/wp-content/uploads/2021/10/Reglement_comites_CEPF_0.pdf": "fr",
    "https://ethrat.stage.mxm.agency/wp-content/uploads/2021/10/Directives_ombudsman_CEPF.pdf": "fr",
    "https://ethrat.stage.mxm.agency/wp-content/uploads/2021/10/Directives_information.pdf": "fr",
    "https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/3.1.4_fondation_usa_en-1.pdf": "en",
    "https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/LEX-3.1.7_annexe2.pdf": "en",
    "https://www.epfl.ch/about/overview/wp-content/uploads/2020/09/LEX-3.1.7_annexe4.pdf": "en",
    "https://ethrat.stage.mxm.agency/wp-content/uploads/2021/10/210101_Richtlinien_NB_ETHR_F.pdf": "fr",
    "https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/4.4.1_Regles_applications_fr_an-1.pdf": "fr", # FIXME : fr + en, grave ?
    "https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/4.4.1_Description_fonction_fr_an.pdf": "fr",
    "https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/LEX-5.1.0.3.pdf": "fr",
    "https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/5.7.1_conv_tresorerie_all.pdf": "fr", # FIXME : fr + de, grave ?
    "https://ethrat.ch/wp-content/uploads/2021/09/Immobilienweisung_ETH-Bereich_2016_F_0.pdf": "fr",
    "https://www.epfl.ch/about/overview/wp-content/uploads/2020/07/LEX-1.1.9.pdf": "fr",
    "https://www.epfl.ch/about/overview/wp-content/uploads/2021/12/LEX-1.1.12.pdf": "fr",
    "https://www.epfl.ch/about/overview/wp-content/uploads/2022/03/LEX-5.1.0.4.pdf": "fr",
    "https://www.epfl.ch/about/overview/wp-content/uploads/2026/01/LEX-1.1.17.pdf": "en",
    "https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/5.7.2_dir_placement_all.pdf": "fr" # FIXME : fr + de, grave ?
}

def detect_language(content, content_format):
    # use directly lib to find language from a text
    if content_format == "txt":
        return detect(content)

    # try to find lang tag in url
    lang_pattern = re.compile(r"[_-](fr|en|an)\.[^/]+$", re.IGNORECASE)
    match = re.search(lang_pattern, content)
    detected_lang = match.group(1) if match else ""
    if detected_lang != "":
        detected_lang = detected_lang.lower()
        detected_lang = "en" if detected_lang == "an" else detected_lang
        return detected_lang

    # use hard-coded correct language for exceptions
    real_lang = hard_coded_langs.get(content)
    if real_lang:
        return real_lang

    # try to detect language from filename -> FIXME : avertir que pas tres fiable ?
    filename_pattern = re.compile(r"[^/]+\.[^/]+$")
    match = re.search(filename_pattern, content)
    filename = match.group() if match else ""
    detected_lang = detect(filename)
    if detected_lang in LANGUAGES:
        return detected_lang

    print(f"Error while trying to detect language for {content}, default to 'fr'")
    return "fr"

for content, metadata_dict in data.items():
    refs = metadata_dict.get("refs")
    metadata = metadata_dict
    if len(refs) == 1:
        metadata["lang"] = refs[0].get("lex_lang")
        data[content] = metadata
    else:
        content_format = metadata_dict.get("content_format")
        metadata["lang"] = detect_language(content, content_format)
        data[content] = metadata

In [6]:
# load all documents

def write_txt(filename, dir, content):
    with open(os.path.join(dir, filename), "w", encoding="utf-8") as f:
        f.write(content)

def download_file(url, dir, filename):
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:147.0) Gecko/20100101 Firefox/147.0"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        with open(os.path.join(dir, filename), "wb") as f:
            f.write(response.content)
    else:
        print(f"Error: the content for {url} can not be fetched (status: {response.status_code})")

shutil.rmtree(Path(DATA_DIR), ignore_errors=True)
os.makedirs(DATA_DIR, exist_ok=True)

for content, metadata in data.items():
    doc_id = metadata.get("doc_id")
    content_format = metadata.get("content_format")
    if content_format == "txt":
        filename = f"{doc_id}.txt"
        write_txt(filename, DATA_DIR, content)
    elif content_format == "docx":
        filename = f"{doc_id}.docx"
        download_file(content, DATA_DIR, filename)
    elif content_format == "pdf":
        filename = f"{doc_id}.pdf"
        download_file(content, DATA_DIR, filename)
    else:
        print(f"File format not yet supported: '{content}'")

In [7]:
# filter pdf to index based on heuristic (np_pages + nb_occ_article) and hard coded values

ARTICLE_PATTERN = re.compile(r"\b(?:Article\s+\d+|Art\.\s*\d+)\b")

stats_per_doc = []

for file in tqdm(DATA_DIR.glob("*.pdf")):
    parsed_file = parser.from_file(str(file))
    metadata = parsed_file.get("metadata", {})
    nb_pages = int(metadata.get("xmpTPg:NPages"))
    extracted_text = parsed_file['content']
    nb_occ_article = sum(1 for _ in ARTICLE_PATTERN.finditer(extracted_text))
    stats_per_doc.append({
        "doc_id": file.stem,
        "nb_pages": nb_pages,
        "nb_occ_article": nb_occ_article
    })

stats = pd.DataFrame(stats_per_doc)
pdfs_not_to_index = stats.loc[(stats["nb_pages"] > 100) | ((stats["nb_pages"] > 50) & (stats["nb_occ_article"] == 0))]["doc_id"].values
print(pdfs_not_to_index)

0it [00:00, ?it/s]

<ArrowStringArray>
[]
Length: 0, dtype: str


In [8]:
# add key 'is_indexed' in metadata dict

for content, metadata_dict in data.items():
    metadata = metadata_dict
    doc_id = metadata["doc_id"]
    if doc_id in pdfs_not_to_index:
        metadata["is_indexed"] = False
    else:
        metadata["is_indexed"] = True
    data[content] = metadata

In [9]:
# load actual state of metadata in json file

os.makedirs(STATS_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
metadata_filename = os.path.join(STATS_DIR, f"{timestamp}_lexes_metadata.json")

with open(metadata_filename, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)